In [4]:
def coin_change(coins, amount):
    """Greedy algorithm to find the minimum number of coins."""
    coins.sort(reverse=True)
    original_amount = amount
    coin_count = 0
    change = []

    for coin in coins:
        while amount >= coin:
            amount -= coin
            coin_count += 1
            change.append(coin)

    if amount == 0:
        print(f"Amount: {original_amount}p")
        print(f"Minimum coins required: {coin_count}")
        print(f"Coins used: {change}\n")
        return coin_count, change
    else:
        print(
            f"Amount: {original_amount}p -> Not possible to get exact change with given coins.\n"
        )
        return None, None


# Executions
uk_coins = [1, 2, 5, 10, 20, 50, 100, 200]
print("--- Exercise 1 & 3: Target 93p ---")
coin_change(uk_coins, 93)

print("--- Exercise 4: Experiments with Different Coins & Sums ---")
coin_change(uk_coins, 147)

custom_coins = [1, 7, 10, 25]
coin_change(custom_coins, 63)

suboptimal_set = [1, 6, 9]
print("Greedy behavior on non-canonical set [1, 6, 9] for 12p:")
coin_change(suboptimal_set, 12)

--- Exercise 1 & 3: Target 93p ---
Amount: 93p
Minimum coins required: 5
Coins used: [50, 20, 20, 2, 1]

--- Exercise 4: Experiments with Different Coins & Sums ---
Amount: 147p
Minimum coins required: 5
Coins used: [100, 20, 20, 5, 2]

Amount: 63p
Minimum coins required: 6
Coins used: [25, 25, 10, 1, 1, 1]

Greedy behavior on non-canonical set [1, 6, 9] for 12p:
Amount: 12p
Minimum coins required: 4
Coins used: [9, 1, 1, 1]



(4, [9, 1, 1, 1])

In [7]:
import sys
import subprocess

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "networkx", "matplotlib"]
)

0

In [8]:
import heapq
import matplotlib.pyplot as plt
import networkx as nx


class Node:

    def __init__(self, name, heuristic_cost):
        self.name = name
        self.heuristic_cost = heuristic_cost
        self.adjacent = {}
        self.parent = None
        self.g_cost = float("inf")

    def add_neighbor(self, neighbor, cost):
        self.adjacent[neighbor] = cost

    def __lt__(self, other):
        return (
            self.g_cost + self.heuristic_cost
            < other.g_cost + other.heuristic_cost
        )


def astar_search(start, goal):
    open_list = []
    closed_set = set()
    start.g_cost = 0
    heapq.heappush(open_list, start)

    while open_list:
        current_node = heapq.heappop(open_list)
        if current_node.name == goal.name:
            path = []
            curr = current_node
            while curr is not None:
                path.append(curr.name)
                curr = curr.parent
            return path[::-1], current_node.g_cost

        closed_set.add(current_node)

        for neighbor, cost in current_node.adjacent.items():
            tentative_g = current_node.g_cost + cost
            if neighbor in closed_set and tentative_g >= neighbor.g_cost:
                continue
            if tentative_g < neighbor.g_cost or neighbor not in open_list:
                neighbor.g_cost = tentative_g
                neighbor.parent = current_node
                if neighbor not in open_list:
                    heapq.heappush(open_list, neighbor)
    return None, float("inf")


# Regional Graph Initialization
nodes = {
    "Sunderland": Node("Sunderland", 6.0),
    "South Shields": Node("South Shields", 8.5),
    "Newcastle": Node("Newcastle", 7.0),
    "Washington": Node("Washington", 0.0),
    "Durham": Node("Durham", 9.0),
    "Peterlee": Node("Peterlee", 14.0),
}

road_edges = [
    ("Sunderland", "South Shields", 7.0),
    ("Sunderland", "Newcastle", 12.0),
    ("Sunderland", "Washington", 6.5),
    ("Sunderland", "Peterlee", 12.0),
    ("South Shields", "Newcastle", 10.0),
    ("Newcastle", "Washington", 8.0),
    ("Washington", "Durham", 11.0),
    ("Durham", "Peterlee", 13.0),
]

for src, dst, dist in road_edges:
    nodes[src].add_neighbor(nodes[dst], dist)
    nodes[dst].add_neighbor(nodes[src], dist)

path, total_cost = astar_search(nodes["Sunderland"], nodes["Washington"])
print("=== A* SEARCH RESULTS ===")
print(f"Optimal Path: {' -> '.join(path)} | Total Distance: {total_cost} miles")

=== A* SEARCH RESULTS ===
Optimal Path: Sunderland -> Washington | Total Distance: 6.5 miles


In [1]:
import random
import string
import time


class GeneticSolver:

    def __init__(self, target, valid_genes, pop_size=150):
        self.target = target
        self.valid_genes = valid_genes
        self.pop_size = pop_size

    def mutate_gene(self):
        return random.choice(self.valid_genes)

    def create_gnome(self):
        return [self.mutate_gene() for _ in range(len(self.target))]

    def calculate_fitness(self, chromosome):
        return sum(
            1 for c1, c2 in zip(chromosome, self.target) if c1 != c2
        )

    def mate(self, parent1, parent2):
        child = []
        for g1, g2 in zip(parent1, parent2):
            prob = random.random()
            if prob < 0.45:
                child.append(g1)
            elif prob < 0.90:
                child.append(g2)
            else:
                child.append(self.mutate_gene())
        return child

    def evolve(self):
        start_time = time.perf_counter()
        population = [
            (self.create_gnome(), 0) for _ in range(self.pop_size)
        ]
        population = [(g, self.calculate_fitness(g)) for g, _ in population]

        generation = 1
        while True:
            population.sort(key=lambda x: x[1])
            if population[0][1] == 0:
                break

            new_generation = []
            elite_count = int(0.10 * self.pop_size)
            new_generation.extend(population[:elite_count])

            mating_pool = population[: int(0.50 * self.pop_size)]
            needed = self.pop_size - elite_count

            for _ in range(needed):
                p1 = random.choice(mating_pool)[0]
                p2 = random.choice(mating_pool)[0]
                child_gnome = self.mate(p1, p2)
                new_generation.append(
                    (child_gnome, self.calculate_fitness(child_gnome))
                )

            population = new_generation
            generation += 1

        elapsed = time.perf_counter() - start_time
        solved_string = "".join(population[0][0])
        return generation, solved_string, elapsed


# Run Benchmark
alpha_genes = string.ascii_letters + " "
ga_ai = GeneticSolver(
    target="Welcome to AI", valid_genes=alpha_genes, pop_size=200
)
gen, text, t_taken = ga_ai.evolve()
print(f"Target: '{text}' | Generations: {gen} | Time: {t_taken:.4f} s")

Target: 'Welcome to AI' | Generations: 19 | Time: 0.0239 s
